[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/01_mcp.ipynb)

*AI Engineering · Notebook 1 of 9*

# MCP: Connecting AI to Tools and Data

The Model Context Protocol from the wire up: JSON-RPC messages, the stdio transport, the initialize handshake, tools, resources and prompts, input-schema validation, errors, pagination, permissions, and turning MCP tools into LLM tool definitions.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** MCP is a universal power socket for AI: build a plug (server) once for a system, and every AI app with the socket can use it.

**What interviewers really test here:** JSON-RPC message shapes, the initialize handshake, tools vs resources vs prompts, stdio vs HTTP, isError results, and MCP security.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why MCP exists *(Basics)*
- [ ] 2. The wire format: JSON-RPC 2.0 *(Basics)*
- [ ] 3. The lifecycle: initialize, then work *(Core)*
- [ ] 4. Tools: schemas, calls and results *(Core)*
- [ ] 5. Build a real server in a few lines *(Core)*
- [ ] 6. Hosts, many servers, and security *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### MCP in one table

| Primitive | Controlled by | Methods |
| --- | --- | --- |
| Tools | the model | `tools/list`, `tools/call` |
| Resources | the application | `resources/list`, `resources/templates/list`, `resources/read`, `resources/subscribe` |
| Prompts | the user | `prompts/list`, `prompts/get` |
| Sampling (client feature) | server asks the host's model | `sampling/createMessage` |
| Roots (client feature) | which folders a server may use | `roots/list` |
| Elicitation (client feature) | server asks the user a question | `elicitation/create` |

### Transports

- **stdio**: the host launches the server process; newline-delimited JSON on stdin/stdout; logs go to stderr. Local, simple, no network.
- **Streamable HTTP**: one HTTP endpoint; requests by POST, responses as JSON or a server-sent-event stream; sessions via a header; authorisation with OAuth 2.1. For remote, multi-user servers.

### MCP vs function calling

**Function calling** is how a model API lets the model request a tool (Claude's `tools` parameter and `tool_use` blocks). **MCP** is how an application *discovers and runs* tools that live in separate servers. A host lists MCP tools, passes them to the model as function definitions, and when the model asks for one, forwards the call to the right server with `tools/call`. They're complementary layers.

---
# Part 2: Lessons, step by step

## Lesson 1: Why MCP exists

*Level: Basics*

Every AI app wants to reach the same things: files, databases, GitHub, Slack, a company API. Before MCP, every app wrote its own connector for every system — **M apps × N systems** integrations. The **Model Context Protocol** is a shared plug: write an **MCP server** for a system once, and any **MCP host** (Claude Desktop, Claude Code, IDEs, your own agent) can use it. That's **M + N**.



| Part | Role | Example |
| --- | --- | --- |
| **Host** | the AI application the user works in | Claude Desktop, an IDE, your agent |
| **Client** | inside the host; one connection per server | created by the host |
| **Server** | exposes tools, resources and prompts | a GitHub server, a Postgres server, your own |



A server offers three kinds of things:




- **Tools** — actions the *model* decides to call (search issues, run a query).
- **Resources** — data the *application* reads and puts in context, addressed by URI (`file:///notes/todo.md`).
- **Prompts** — reusable templates the *user* picks (a "/summarise" command).

**✏️ Your turn:** you now have what you need for [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 2: The wire format: JSON-RPC 2.0

*Level: Basics*

Every MCP message is a small JSON object. There are three shapes:



```python
{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}                         # request: expects a reply
{"jsonrpc": "2.0", "id": 1, "result": {"tools": [...]}}                     # response with the same id
{"jsonrpc": "2.0", "method": "notifications/initialized"}                   # notification: no id, no reply
```



Errors use standard codes: **-32700** parse error, **-32600** invalid request, **-32601** method not found, **-32602** invalid params, **-32603** internal error.



**Transports** carry these messages. With **stdio**, the host starts the server as a child process and they exchange one JSON message per line on stdin/stdout — simple and private, ideal for local tools. With **Streamable HTTP**, the server runs as a web service that many clients can reach, with authentication (OAuth) — used for remote servers.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def wire_demo():
    req = {"jsonrpc": "2.0", "id": 7, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 2, "b": 3}}}
    line = json.dumps(req)
    received = json.loads(line)
    reply = {"jsonrpc": "2.0", "id": received["id"],
             "result": {"content": [{"type": "text", "text": "5"}], "isError": False}}


wire_demo()

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 3: The lifecycle: initialize, then work

*Level: Core*

1. The client sends `initialize` with the protocol version it wants, its capabilities (e.g. `sampling`, `roots`) and its name.
2. The server replies with the version it will use and *its* capabilities (`tools`, `resources`, `prompts`, `logging`).
3. The client sends `notifications/initialized`. Now normal work starts: `tools/list`, `tools/call`, `resources/read`, `prompts/get` …



Capabilities matter: a client must not call `prompts/list` on a server that didn't declare `prompts`. Long lists are **paginated** with an opaque `nextCursor`. Long tools can send `notifications/progress`, and either side can cancel a request.

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q16](#q16), [Q11](#q11), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Tools: schemas, calls and results

*Level: Core*

A tool is a name, a description the model reads to decide when to use it, and an `inputSchema` (JSON Schema) for its arguments:



```python
{"name": "get_weather", "description": "Forecast for a city.",
 "inputSchema": {"type": "object", "properties": {"city": {"type": "string"}, "days": {"type": "integer"}},
                 "required": ["city"]}}
```



A call returns `content` blocks (text, images, resource links) and `isError`. A tool that fails returns `isError: true` with a readable message *as a normal result*, so the model can see what went wrong and try again; protocol-level errors are for broken requests.



**Writing good tools is prompt engineering**: clear names, a description that says when to use it, few parameters with enums where possible, and short outputs — every token a tool returns lands in the model's context.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q14](#q14), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 5: Build a real server in a few lines

*Level: Core*

The official Python SDK (`pip install mcp`) turns type-hinted functions into tools, exactly like the questions here do by hand:



```python
from mcp.server.fastmcp import FastMCP      # SDK v1; in v2 the class is MCPServer in mcp.server.mcpserver

mcp = FastMCP("weather")

@mcp.tool()
def get_weather(city: str, days: int = 1) -> str:
    """Forecast for a city."""
    return f"{city}: sunny for {days} day(s)"

@mcp.resource("notes://{name}")
def note(name: str) -> str:
    return open(f"notes/{name}.md").read()

if __name__ == "__main__":
    mcp.run()          # stdio transport by default
```



Connect it to Claude Code with `claude mcp add weather -- python server.py`, or add it to Claude Desktop's config file. The **MCP Inspector** (`npx @modelcontextprotocol/inspector`) lets you call your tools by hand while you develop. SDK details change between versions — check the docs for the one you install.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q15](#q15), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 6: Hosts, many servers, and security

*Level: Advanced*

A host usually connects to **several** servers and merges their tools into one list for the model — so it must avoid name clashes (prefix with the server name) and convert MCP tool definitions into the model API's format.



Security is the hard part, because tools act with the user's permissions:




- **Human approval** for anything that writes or deletes. Tool **annotations** (`readOnlyHint`, `destructiveHint`) help a host decide, but they are hints from the server, not guarantees.
- **Prompt injection**: text returned by a tool (a web page, an email, an issue comment) can contain instructions. Treat tool output as data, never as commands.
- **Least privilege**: give each server only the tokens and folders it needs; prefer read-only by default.
- **Only install servers you trust** — a local server runs code on your machine.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
def get_weather(city: str, days: int = 1) -> str:
    """Forecast for a city.

    Longer notes the model doesn't need."""
    return f"{city}: sunny for {days} day(s)"

def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

REGISTRY = {"add": add, "get_weather": get_weather}


<a id="q1"></a>
### Q1. Write a JSON-RPC Request

🟢 Warm-up · *Message shape*

Return the JSON text of a JSON-RPC 2.0 request with keys in this order: `jsonrpc`, `id`, `method`, `params` (omit `params` if it is None).

**Examples**

- `msg_id = 1, method = 'tools/list', params = None` → `'{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}'`
- `msg_id = 2, method = 'tools/call', params = {'name': 'add', 'arguments': {'a': 2, 'b': 3}}` → `'{"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 2, "b": 3}}}'`

In [ ]:
def request(msg_id, method, params):
    """
    Return the JSON text of a JSON-RPC 2.0 request with keys in this order:
    jsonrpc, id, method, params (omit params if it is None).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(request, [
    ((1, 'tools/list', None), '{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}'),
    ((2, 'tools/call', {'name': 'add', 'arguments': {'a': 2, 'b': 3}}), '{"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 2, "b": 3}}}'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the text by hand with string pieces.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a dict and `json.dumps` it — insertion order is kept.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Build the text by hand with string pieces.

```python
def request(msg_id, method, params):
    text = '{"jsonrpc": "2.0", "id": ' + str(msg_id) + ', "method": "' + method + '"'
    if params is not None:
        text += ', "params": ' + json.dumps(params)
    return text + "}"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Build a dict and `json.dumps` it — insertion order is kept.

```python
def request(msg_id, method, params):
    msg = {"jsonrpc": "2.0", "id": msg_id, "method": method}
    if params is not None:
        msg["params"] = params
    return json.dumps(msg)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the text by hand with string pieces. That costs O(n) time and O(n) space."

"I can do better: Build a dict and `json.dumps` it — insertion order is kept. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/write-a-json-rpc-request)

<a id="q2"></a>
### Q2. Read Messages from stdio

🟢 Warm-up · *Newline-delimited JSON*

The stdio transport sends one JSON message per line. Given the raw text read from stdout, return the `method` of each message that has one (skip blank lines and responses).

**Examples**

- `stream = '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{}}\n\n{"jsonrpc":"2.0","method":"notifications/initialized"}\n{"jsonrpc":"2.0","id":1,"result":{}}\n'` → `['initialize', 'notifications/initialized']`

In [ ]:
def methods(stream):
    """
    The stdio transport sends one JSON message per line. Given the raw text read
    from stdout, return the method of each message that has one (skip blank
    lines and responses).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(methods, [
    (('{"jsonrpc":"2.0","id":1,"method":"initialize","params":{}}\n\n{"jsonrpc":"2.0","method":"notifications/initialized"}\n{"jsonrpc":"2.0","id":1,"result":{}}\n',), ['initialize', 'notifications/initialized']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on newlines, skip empties, parse each and collect methods.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension over `splitlines()`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split on newlines, skip empties, parse each and collect methods.

```python
def methods(stream):
    out = []
    for line in stream.split("\n"):
        line = line.strip()
        if line == "":
            continue
        msg = json.loads(line)
        if "method" in msg:
            out.append(msg["method"])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension over `splitlines()`.

```python
def methods(stream):
    msgs = [json.loads(l) for l in stream.splitlines() if l.strip()]
    return [m["method"] for m in msgs if "method" in m]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on newlines, skip empties, parse each and collect methods. That costs O(n) time and O(n) space."

"I can do better: A comprehension over `splitlines()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/read-messages-from-stdio)

<a id="q3"></a>
### Q3. What Kind of Message Is This?

🟢 Warm-up · *Requests, notifications, responses*

Return 'request' (method and id), 'notification' (method, no id), 'result' (id and result), 'error' (id and error) or 'invalid'.

**Examples**

- `msg = {'jsonrpc': '2.0', 'id': 3, 'method': 'tools/list'}` → `'request'`
- `msg = {'jsonrpc': '2.0', 'method': 'notifications/progress'}` → `'notification'`
- `msg = {'jsonrpc': '2.0', 'id': 3, 'result': {}}` → `'result'`

In [ ]:
def kind(msg):
    """
    Return 'request' (method and id), 'notification' (method, no id), 'result'
    (id and result), 'error' (id and error) or 'invalid'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kind, [
    (({'jsonrpc': '2.0', 'id': 3, 'method': 'tools/list'},), 'request'),
    (({'jsonrpc': '2.0', 'method': 'notifications/progress'},), 'notification'),
    (({'jsonrpc': '2.0', 'id': 3, 'result': {}},), 'result'),
    (({'jsonrpc': '2.0', 'id': 3, 'error': {'code': -32601}},), 'error'),
    (({'id': 1},), 'invalid'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain on which keys exist.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Map the set of present keys to a kind with a small lookup table.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain on which keys exist.

```python
def kind(msg):
    if "method" in msg and "id" in msg:
        return "request"
    if "method" in msg:
        return "notification"
    if "id" in msg and "result" in msg:
        return "result"
    if "id" in msg and "error" in msg:
        return "error"
    return "invalid"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Map the set of present keys to a kind with a small lookup table.

```python
def kind(msg):
    shape = frozenset(k for k in ("id", "method", "result", "error") if k in msg)
    table = {frozenset({"id", "method"}): "request", frozenset({"method"}): "notification",
             frozenset({"id", "result"}): "result", frozenset({"id", "error"}): "error"}
    return table.get(shape, "invalid")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain on which keys exist. That costs O(1) time and O(1) space."

"I can do better: Map the set of present keys to a kind with a small lookup table. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/what-kind-of-message-is-this)

<a id="q4"></a>
### Q4. Answer With the Right Error

🟢 Warm-up · *JSON-RPC error codes*

A server receives `raw` text and knows `methods`. Return the error code to send back, or 0 if the request is fine: -32700 if it isn't JSON, -32600 if it has no `method` string, -32601 if the method is unknown.

**Examples**

- `raw = '{"jsonrpc":"2.0","id":1,"method":"tools/list"}', methods = ['tools/list', 'tools/call']` → `0`
- `raw = '{oops', methods = ['tools/list']` → `-32700`
- `raw = '{"jsonrpc":"2.0","id":1}', methods = ['tools/list']` → `-32600`

In [ ]:
ERRORS = {"parse": (-32700, "Parse error"), "invalid": (-32600, "Invalid Request"),
          "method": (-32601, "Method not found"), "params": (-32602, "Invalid params"), "internal": (-32603, "Internal error")}

def error_code(raw, methods):
    """
    A server receives raw text and knows methods. Return the error code to send
    back, or 0 if the request is fine: -32700 if it isn't JSON, -32600 if it has
    no method string, -32601 if the method is unknown.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(error_code, [
    (('{"jsonrpc":"2.0","id":1,"method":"tools/list"}', ['tools/list', 'tools/call']), 0),
    (('{oops', ['tools/list']), -32700),
    (('{"jsonrpc":"2.0","id":1}', ['tools/list']), -32600),
    (('{"jsonrpc":"2.0","id":1,"method":"tools/delete"}', ['tools/list']), -32601),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested try / if.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same checks, using the named `ERRORS` table so codes aren't magic numbers.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested try / if.

```python
def error_code(raw, methods):
    try:
        msg = json.loads(raw)
    except ValueError:
        return -32700
    if not isinstance(msg, dict) or not isinstance(msg.get("method"), str):
        return -32600
    if msg["method"] not in methods:
        return -32601
    return 0
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same checks, using the named `ERRORS` table so codes aren't magic numbers.

```python
def error_code(raw, methods):
    try:
        msg = json.loads(raw)
    except json.JSONDecodeError:
        return ERRORS["parse"][0]
    if not isinstance(msg, dict) or not isinstance(msg.get("method"), str):
        return ERRORS["invalid"][0]
    return 0 if msg["method"] in methods else ERRORS["method"][0]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested try / if. That costs O(n) time and O(n) space."

"I can do better: Same checks, using the named `ERRORS` table so codes aren't magic numbers. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/answer-with-the-right-error)

<a id="q5"></a>
### Q5. Agree on a Protocol Version

🟢 Warm-up · *initialize*

In `initialize` the client proposes a protocol version. The server answers with the same version if it supports it, otherwise with its newest supported version. Versions are dates like '2025-06-18'.

**Examples**

- `requested = '2025-06-18', supported = ['2024-11-05', '2025-03-26', '2025-06-18']` → `'2025-06-18'`
- `requested = '2026-01-01', supported = ['2024-11-05', '2025-06-18']` → `'2025-06-18'`

In [ ]:
def negotiate(requested, supported):
    """
    In initialize the client proposes a protocol version. The server answers
    with the same version if it supports it, otherwise with its newest supported
    version. Versions are dates like '2025-06-18'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(negotiate, [
    (('2025-06-18', ['2024-11-05', '2025-03-26', '2025-06-18']), '2025-06-18'),
    (('2026-01-01', ['2024-11-05', '2025-06-18']), '2025-06-18'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop to check membership and find the max.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dates in ISO form sort as text, so `max` works directly.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop to check membership and find the max.

```python
def negotiate(requested, supported):
    newest = supported[0]
    for v in supported:
        if v == requested:
            return v
        if v > newest:
            newest = v
    return newest
```

**Cost:** O(v) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Dates in ISO form sort as text, so `max` works directly.

```python
def negotiate(requested, supported):
    return requested if requested in supported else max(supported)
```

**Cost:** O(v) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop to check membership and find the max. That costs O(v) time and O(1) space."

"I can do better: Dates in ISO form sort as text, so `max` works directly. That's O(v) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/agree-on-a-protocol-version)

<a id="q6"></a>
### Q6. Which Features Can Be Used?

🟢 Warm-up · *Capability negotiation*

Client and server each list capabilities. A feature is usable if the side that must provide it declares it: tools, resources, prompts and logging come from the **server**; sampling, roots and elicitation from the **client**. Return the usable features, sorted.

**Examples**

- `client_caps = ['sampling', 'roots'], server_caps = ['tools', 'resources', 'logging']` → `['logging', 'resources', 'roots', 'sampling', 'tools']`
- `client_caps = [], server_caps = ['tools']` → `['tools']`

In [ ]:
def usable(client_caps, server_caps):
    """
    Client and server each list capabilities. A feature is usable if the side
    that must provide it declares it: tools, resources, prompts and logging come
    from the server; sampling, roots and elicitation from the client. Return the
    usable features, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(usable, [
    ((['sampling', 'roots'], ['tools', 'resources', 'logging']), ['logging', 'resources', 'roots', 'sampling', 'tools']),
    (([], ['tools']), ['tools']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each known feature against the right side in an if-chain.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two sets and an intersection each.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each known feature against the right side in an if-chain.

```python
def usable(client_caps, server_caps):
    out = []
    for f in ["tools", "resources", "prompts", "logging"]:
        if f in server_caps:
            out.append(f)
    for f in ["sampling", "roots", "elicitation"]:
        if f in client_caps:
            out.append(f)
    return sorted(out)
```

**Cost:** O(f) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two sets and an intersection each.

```python
def usable(client_caps, server_caps):
    server_side, client_side = {"tools", "resources", "prompts", "logging"}, {"sampling", "roots", "elicitation"}
    return sorted((server_side & set(server_caps)) | (client_side & set(client_caps)))
```

**Cost:** O(f) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each known feature against the right side in an if-chain. That costs O(f) time and O(f) space."

"I can do better: Two sets and an intersection each. That's O(f) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/which-features-can-be-used)

<a id="q7"></a>
### Q7. Read a Resource

🟢 Warm-up · *resources/read*

Return the `resources/read` result for a text file: `{'contents': [{'uri': uri, 'mimeType': …, 'text': text}]}` with the MIME type from the extension (.md → text/markdown, .json → application/json, .csv → text/csv, otherwise text/plain).

**Examples**

- `uri = 'file:///docs/guide.md', text = '# Hi'` → `{'contents': [{'uri': 'file:///docs/guide.md', 'mimeType': 'text/markdown', 'text': '# Hi'}]}`
- `uri = 'file:///data/x.parquet', text = '...'` → `{'contents': [{'uri': 'file:///data/x.parquet', 'mimeType': 'text/plain', 'text': '...'}]}`

In [ ]:
def read_resource(uri, text):
    """
    Return the resources/read result for a text file: {'contents': [{'uri': uri,
    'mimeType': …, 'text': text}]} with the MIME type from the extension (.md →
    text/markdown, .json → application/json, .csv → text/csv, otherwise
    text/plain).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_resource, [
    (('file:///docs/guide.md', '# Hi'), {'contents': [{'uri': 'file:///docs/guide.md', 'mimeType': 'text/markdown', 'text': '# Hi'}]}),
    (('file:///data/x.parquet', '...'), {'contents': [{'uri': 'file:///data/x.parquet', 'mimeType': 'text/plain', 'text': '...'}]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain on the extension.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict lookup on the suffix.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain on the extension.

```python
def read_resource(uri, text):
    if uri.endswith(".md"):
        mime = "text/markdown"
    elif uri.endswith(".json"):
        mime = "application/json"
    elif uri.endswith(".csv"):
        mime = "text/csv"
    else:
        mime = "text/plain"
    return {"contents": [{"uri": uri, "mimeType": mime, "text": text}]}
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict lookup on the suffix.

```python
def read_resource(uri, text):
    mime = {"md": "text/markdown", "json": "application/json", "csv": "text/csv"}.get(uri.rsplit(".", 1)[-1], "text/plain")
    return {"contents": [{"uri": uri, "mimeType": mime, "text": text}]}
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain on the extension. That costs O(1) time and O(1) space."

"I can do better: A dict lookup on the suffix. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/read-a-resource)

<a id="q8"></a>
### Q8. MCP Tools → Claude API Tools

🟢 Warm-up · *Bridging protocols*

An MCP client hands tools to the model. Convert MCP tool definitions to Claude API tool definitions: same `name` and `description`, and `inputSchema` becomes `input_schema`.

**Examples**

- `mcp_tools = [{'name': 'add', 'description': 'Add two numbers.', 'inputSchema': {'type': 'object', 'properties': {}}}]` → `[{'name': 'add', 'description': 'Add two numbers.', 'input_schema': {'type': 'object', 'properties': {}}}]`

In [ ]:
def to_claude_tools(mcp_tools):
    """
    An MCP client hands tools to the model. Convert MCP tool definitions to
    Claude API tool definitions: same name and description, and inputSchema
    becomes input_schema.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_claude_tools, [
    (([{'name': 'add', 'description': 'Add two numbers.', 'inputSchema': {'type': 'object', 'properties': {}}}],), [{'name': 'add', 'description': 'Add two numbers.', 'input_schema': {'type': 'object', 'properties': {}}}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build each new dict field by field.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Build each new dict field by field.

```python
def to_claude_tools(mcp_tools):
    out = []
    for t in mcp_tools:
        out.append({"name": t["name"], "description": t["description"], "input_schema": t["inputSchema"]})
    return out
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def to_claude_tools(mcp_tools):
    return [{"name": t["name"], "description": t["description"], "input_schema": t["inputSchema"]} for t in mcp_tools]
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build each new dict field by field. That costs O(t) time and O(t) space."

"I can do better: A comprehension. That's O(t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/mcp-tools-claude-api-tools)

<a id="q9"></a>
### Q9. Fill a Prompt Template

🟢 Warm-up · *prompts/get*

Return the `prompts/get` result: `{'messages': [{'role': 'user', 'content': {'type': 'text', 'text': filled}}]}`. If an argument is missing, return `{'error': 'missing: name'}` for the first missing one.

**Examples**

- `name = 'summarise', arguments = {'topic': 'MCP', 'length': '3'}` → `{'messages': [{'role': 'user', 'content': {'type': 'text', 'text': 'Summarise MCP in 3 bullet points.'}}]}`
- `name = 'summarise', arguments = {'topic': 'MCP'}` → `{'error': 'missing: length'}`

In [ ]:
PROMPTS = {"summarise": {"arguments": ["topic", "length"], "template": "Summarise {topic} in {length} bullet points."}}

def get_prompt(name, arguments):
    """
    Return the prompts/get result: {'messages': [{'role': 'user', 'content':
    {'type': 'text', 'text': filled}}]}. If an argument is missing, return
    {'error': 'missing: name'} for the first missing one.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(get_prompt, [
    (('summarise', {'topic': 'MCP', 'length': '3'}), {'messages': [{'role': 'user', 'content': {'type': 'text', 'text': 'Summarise MCP in 3 bullet points.'}}]}),
    (('summarise', {'topic': 'MCP'}), {'error': 'missing: length'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Replace each {name} by hand after checking arguments.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`str.format(**arguments)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Replace each {name} by hand after checking arguments.

```python
def get_prompt(name, arguments):
    spec = PROMPTS[name]
    for a in spec["arguments"]:
        if a not in arguments:
            return {"error": "missing: " + a}
    text = spec["template"]
    for k, v in arguments.items():
        text = text.replace("{" + k + "}", v)
    return {"messages": [{"role": "user", "content": {"type": "text", "text": text}}]}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `str.format(**arguments)`.

```python
def get_prompt(name, arguments):
    spec = PROMPTS[name]
    missing = [a for a in spec["arguments"] if a not in arguments]
    if missing:
        return {"error": f"missing: {missing[0]}"}
    return {"messages": [{"role": "user", "content": {"type": "text", "text": spec["template"].format(**arguments)}}]}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Replace each {name} by hand after checking arguments. That costs O(n) time and O(n) space."

"I can do better: `str.format(**arguments)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/fill-a-prompt-template)

<a id="q10"></a>
### Q10. Keep Tool Output Within Budget

🟢 Warm-up · *Context hygiene*

Tool results go into the model's context. If `text` is longer than `max_tokens` (estimate 4 characters per token), keep the first part and append `'\n…[truncated N chars]'` where N is the number of characters removed.

**Examples**

- `text = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx', max_tokens = 10` → `'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx\n…[truncated 10 chars]'`
- `text = 'short', max_tokens = 10` → `'short'`

In [ ]:
def truncate(text, max_tokens):
    """
    Tool results go into the model's context. If text is longer than max_tokens
    (estimate 4 characters per token), keep the first part and append
    '\n…[truncated N chars]' where N is the number of characters removed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(truncate, [
    (('xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx', 10), 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx\n…[truncated 10 chars]'),
    (('short', 10), 'short'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the limit and build the string step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression with slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute the limit and build the string step by step.

```python
def truncate(text, max_tokens):
    limit = max_tokens * 4
    if len(text) <= limit:
        return text
    removed = len(text) - limit
    return text[:limit] + "\n…[truncated " + str(removed) + " chars]"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression with slicing.

```python
def truncate(text, max_tokens):
    limit = max_tokens * 4
    return text if len(text) <= limit else f"{text[:limit]}\n…[truncated {len(text) - limit} chars]"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the limit and build the string step by step. That costs O(n) time and O(n) space."

"I can do better: A conditional expression with slicing. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/keep-tool-output-within-budget)

<a id="q11"></a>
### Q11. Progress Notifications

🟢 Warm-up · *Long-running tools*

A long tool sends `notifications/progress` with `progress` and `total`. Return the percentage (rounded down) after each notification, ignoring any that go backwards.

**Examples**

- `notes = [(10, 200), (50, 200), (40, 200), (200, 200)]` → `[5, 25, 100]`

In [ ]:
def percents(notes):
    """
    A long tool sends notifications/progress with progress and total. Return the
    percentage (rounded down) after each notification, ignoring any that go
    backwards.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(percents, [
    (([(10, 200), (50, 200), (40, 200), (200, 200)],), [5, 25, 100]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the best progress so far in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with integer division; keep the last accepted value.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the best progress so far in a loop.

```python
def percents(notes):
    out, best = [], -1
    for progress, total in notes:
        if progress < best:
            continue
        best = progress
        out.append(int(progress / total * 100))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with integer division; keep the last accepted value.

```python
def percents(notes):
    out, best = [], -1
    for progress, total in notes:
        if progress >= best:
            best = progress
            out.append(progress * 100 // total)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the best progress so far in a loop. That costs O(n) time and O(n) space."

"I can do better: Same loop with integer division; keep the last accepted value. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/progress-notifications)

<a id="q12"></a>
### Q12. Validate Tool Arguments

🟡 Interview standard · *inputSchema*

Check `args` against a tool's JSON-Schema `inputSchema` (types string / integer / number / boolean, `enum`, `required`, no unknown keys). Return a sorted list of problems like `'missing: city'`, `'type: days'`, `'enum: units'`, `'unknown: x'`.

**Examples**

- `schema = {'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, args = {'city': 'Pune', 'days': 3}` → `[]`
- `schema = {'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, args = {'days': '3', 'units': 'kelvin', 'x': 1}` → `['enum: units', 'missing: city', 'type: days', 'unknown: x']`
- `schema = {'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, args = {'city': 'Goa', 'days': True}` → `['type: days']`

In [ ]:
def validate_args(schema, args):
    """
    Check args against a tool's JSON-Schema inputSchema (types string / integer
    / number / boolean, enum, required, no unknown keys). Return a sorted list
    of problems like 'missing: city', 'type: days', 'enum: units', 'unknown: x'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(validate_args, [
    (({'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, {'city': 'Pune', 'days': 3}), []),
    (({'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, {'days': '3', 'units': 'kelvin', 'x': 1}), ['enum: units', 'missing: city', 'type: days', 'unknown: x']),
    (({'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}, 'units': {'type': 'string', 'enum': ['metric', 'imperial']}}, 'required': ['city']}, {'city': 'Goa', 'days': True}), ['type: days']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the schema and the arguments with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A type table `{'string': str, …}`; remember `bool` is a subclass of `int` in Python.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over the schema and the arguments with if-statements.

```python
def validate_args(schema, args):
    problems = []
    props = schema["properties"]
    for name in schema.get("required", []):
        if name not in args:
            problems.append("missing: " + name)
    for name, value in args.items():
        if name not in props:
            problems.append("unknown: " + name)
            continue
        t = props[name]["type"]
        ok = False
        if t == "string":
            ok = isinstance(value, str)
        elif t == "integer":
            ok = isinstance(value, int) and not isinstance(value, bool)
        elif t == "number":
            ok = isinstance(value, (int, float)) and not isinstance(value, bool)
        elif t == "boolean":
            ok = isinstance(value, bool)
        if not ok:
            problems.append("type: " + name)
        elif "enum" in props[name] and value not in props[name]["enum"]:
            problems.append("enum: " + name)
    return sorted(problems)
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A type table `{'string': str, …}`; remember `bool` is a subclass of `int` in Python.

```python
def validate_args(schema, args):
    types = {"string": (str,), "integer": (int,), "number": (int, float), "boolean": (bool,)}
    props = schema["properties"]
    problems = [f"missing: {n}" for n in schema.get("required", []) if n not in args]
    for name, value in args.items():
        spec = props.get(name)
        if spec is None:
            problems.append(f"unknown: {name}")
        elif not isinstance(value, types[spec["type"]]) or (isinstance(value, bool) and spec["type"] != "boolean"):
            problems.append(f"type: {name}")
        elif value not in spec.get("enum", [value]):
            problems.append(f"enum: {name}")
    return sorted(problems)
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the schema and the arguments with if-statements. That costs O(k) time and O(k) space."

"I can do better: A type table `{'string': str, …}`; remember `bool` is a subclass of `int` in Python. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/validate-tool-arguments)

<a id="q13"></a>
### Q13. Describe Python Functions as Tools

🟡 Interview standard · *tools/list from code*

Build the `tools/list` entries for the Python functions named in `fn_names` (look them up in `FUNCS`), like an MCP SDK's `@tool` decorator does: name, the docstring's first line as description, and an `inputSchema` from the type hints (str → string, int → integer, float → number, bool → boolean). Parameters without defaults are required.

**Examples**

- `fn_names = ['get_weather', 'add']` → `[{'name': 'get_weather', 'description': 'Forecast for a city.', 'inputSchema': {'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}}, 'required': ['city']}}, {'name': 'add', 'description': 'Add two numbers.', 'inputSchema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}}]`

In [ ]:
import inspect

FUNCS = {"get_weather": get_weather, "add": add}

def describe_tools(fn_names):
    """
    Build the tools/list entries for the Python functions named in fn_names
    (look them up in FUNCS), like an MCP SDK's @tool decorator does: name, the
    docstring's first line as description, and an inputSchema from the type
    hints (str → string, int → integer, float → number, bool → boolean).
    Parameters without defaults are required.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(describe_tools, [
    ((['get_weather', 'add'],), [{'name': 'get_weather', 'description': 'Forecast for a city.', 'inputSchema': {'type': 'object', 'properties': {'city': {'type': 'string'}, 'days': {'type': 'integer'}}, 'required': ['city']}}, {'name': 'add', 'description': 'Add two numbers.', 'inputSchema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Read `__annotations__`, `__defaults__` and the code object's argument names by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`inspect.signature` gives every parameter with its annotation and default.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Read `__annotations__`, `__defaults__` and the code object's argument names by hand.

```python
def describe_tools(fn_names):
    names = {str: "string", int: "integer", float: "number", bool: "boolean"}
    tools = []
    for f in [FUNCS[n] for n in fn_names]:
        argnames = f.__code__.co_varnames[:f.__code__.co_argcount]
        n_defaults = len(f.__defaults__ or ())
        props, required = {}, []
        for i, arg in enumerate(argnames):
            props[arg] = {"type": names[f.__annotations__[arg]]}
            if i < len(argnames) - n_defaults:
                required.append(arg)
        tools.append({"name": f.__name__, "description": f.__doc__.strip().split("\n")[0],
                      "inputSchema": {"type": "object", "properties": props, "required": required}})
    return tools
```

**Cost:** O(params) time, O(params) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `inspect.signature` gives every parameter with its annotation and default.

```python
def describe_tools(fn_names):
    names = {str: "string", int: "integer", float: "number", bool: "boolean"}
    tools = []
    for f in map(FUNCS.get, fn_names):
        params = inspect.signature(f).parameters.values()
        tools.append({
            "name": f.__name__,
            "description": inspect.getdoc(f).splitlines()[0],
            "inputSchema": {"type": "object",
                            "properties": {p.name: {"type": names[p.annotation]} for p in params},
                            "required": [p.name for p in params if p.default is inspect.Parameter.empty]},
        })
    return tools
```

**Cost:** O(params) time, O(params) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Read `__annotations__`, `__defaults__` and the code object's argument names by hand. That costs O(params) time and O(params) space."

"I can do better: `inspect.signature` gives every parameter with its annotation and default. That's O(params) time and O(params) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/describe-python-functions-as-tools)

<a id="q14"></a>
### Q14. Handle tools/call

🟡 Interview standard · *Results and isError*

Run the named tool from `registry` with the arguments and return an MCP result: `{'content': [{'type': 'text', 'text': str(value)}], 'isError': False}`. Unknown tools or exceptions return the error text with `isError: True` — tool failures are results the model can read, not protocol errors.

**Examples**

- `params = {'name': 'add', 'arguments': {'a': 2, 'b': 3.5}}` → `{'content': [{'type': 'text', 'text': '5.5'}], 'isError': False}`
- `params = {'name': 'add', 'arguments': {'a': 1}}` → `{'content': [{'type': 'text', 'text': "TypeError: add() missing 1 required positional argument: 'b'"}], 'isError': True}`
- `params = {'name': 'rm', 'arguments': {}}` → `{'content': [{'type': 'text', 'text': 'Unknown tool: rm'}], 'isError': True}`

In [ ]:
def call_tool(params):
    """
    Run the named tool from registry with the arguments and return an MCP
    result: {'content': [{'type': 'text', 'text': str(value)}], 'isError':
    False}. Unknown tools or exceptions return the error text with isError: True
    — tool failures are results the model can read, not protocol errors.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(call_tool, [
    (({'name': 'add', 'arguments': {'a': 2, 'b': 3.5}},), {'content': [{'type': 'text', 'text': '5.5'}], 'isError': False}),
    (({'name': 'add', 'arguments': {'a': 1}},), {'content': [{'type': 'text', 'text': "TypeError: add() missing 1 required positional argument: 'b'"}], 'isError': True}),
    (({'name': 'rm', 'arguments': {}},), {'content': [{'type': 'text', 'text': 'Unknown tool: rm'}], 'isError': True}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Look up, call, catch, and build the dict in two branches.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A small helper that formats both outcomes the same way.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Look up, call, catch, and build the dict in two branches.

```python
def call_tool(params):
    name = params["name"]
    if name not in REGISTRY:
        return {"content": [{"type": "text", "text": "Unknown tool: " + name}], "isError": True}
    try:
        value = REGISTRY[name](**params.get("arguments", {}))
    except Exception as e:
        return {"content": [{"type": "text", "text": type(e).__name__ + ": " + str(e)}], "isError": True}
    return {"content": [{"type": "text", "text": str(value)}], "isError": False}
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A small helper that formats both outcomes the same way.

```python
def call_tool(params):
    def result(text, is_error):
        return {"content": [{"type": "text", "text": text}], "isError": is_error}
    fn = REGISTRY.get(params["name"])
    if fn is None:
        return result(f"Unknown tool: {params['name']}", True)
    try:
        return result(str(fn(**params.get("arguments", {}))), False)
    except Exception as e:
        return result(f"{type(e).__name__}: {e}", True)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Look up, call, catch, and build the dict in two branches. That costs O(1) time and O(1) space."

"I can do better: A small helper that formats both outcomes the same way. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/handle-tools-call)

<a id="q15"></a>
### Q15. Route a Resource URI

🟡 Interview standard · *Resource templates*

Resources are addressed by URI. Given templates like `'notes://{folder}/{name}'`, return `(template, parameters)` for the first template that matches `uri`, or None. Parameters match one path segment (no '/').

**Examples**

- `templates = ['notes://{folder}/{name}', 'db://customers/{id}'], uri = 'db://customers/C042'` → `('db://customers/{id}', {'id': 'C042'})`
- `templates = ['notes://{folder}/{name}'], uri = 'notes://work/todo.md'` → `('notes://{folder}/{name}', {'folder': 'work', 'name': 'todo.md'})`
- `templates = ['notes://{folder}/{name}'], uri = 'notes://a/b/c'` → `None`

In [ ]:
import re

def route(templates, uri):
    """
    Resources are addressed by URI. Given templates like
    'notes://{folder}/{name}', return (template, parameters) for the first
    template that matches uri, or None. Parameters match one path segment (no
    '/').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(route, [
    ((['notes://{folder}/{name}', 'db://customers/{id}'], 'db://customers/C042'), ('db://customers/{id}', {'id': 'C042'})),
    ((['notes://{folder}/{name}'], 'notes://work/todo.md'), ('notes://{folder}/{name}', {'folder': 'work', 'name': 'todo.md'})),
    ((['notes://{folder}/{name}'], 'notes://a/b/c'), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split both on '/' and compare segment by segment.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Turn each template into a regex with named groups.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split both on '/' and compare segment by segment.

```python
def route(templates, uri):
    for t in templates:
        a, b = t.split("/"), uri.split("/")
        if len(a) != len(b):
            continue
        params, ok = {}, True
        for x, y in zip(a, b):
            if x.startswith("{") and x.endswith("}"):
                params[x[1:-1]] = y
            elif x != y:
                ok = False
                break
        if ok:
            return (t, params)
    return None
```

**Cost:** O(t · s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Turn each template into a regex with named groups.

```python
def route(templates, uri):
    for t in templates:
        pattern = re.sub(r"\\\{(\w+)\\\}", r"(?P<\1>[^/]+)", re.escape(t))
        m = re.fullmatch(pattern, uri)
        if m:
            return (t, m.groupdict())
    return None
```

**Cost:** O(t · s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split both on '/' and compare segment by segment. That costs O(t · s) time and O(s) space."

"I can do better: Turn each template into a regex with named groups. That's O(t · s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/route-a-resource-uri)

<a id="q16"></a>
### Q16. Paginate tools/list

🟡 Interview standard · *Cursors*

A server with many tools returns them in pages of `size`. The cursor is the index (as text) of the next tool. Return the page for `cursor` (None = start) as `{'tools': [...names], 'nextCursor': … or None}`.

**Examples**

- `tools = ['a', 'b', 'c', 'd', 'e'], size = 2, cursor = None` → `{'tools': ['a', 'b'], 'nextCursor': '2'}`
- `tools = ['a', 'b', 'c', 'd', 'e'], size = 2, cursor = '4'` → `{'tools': ['e'], 'nextCursor': None}`

In [ ]:
def page(tools, size, cursor):
    """
    A server with many tools returns them in pages of size. The cursor is the
    index (as text) of the next tool. Return the page for cursor (None = start)
    as {'tools': [...names], 'nextCursor': … or None}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(page, [
    ((['a', 'b', 'c', 'd', 'e'], 2, None), {'tools': ['a', 'b'], 'nextCursor': '2'}),
    ((['a', 'b', 'c', 'd', 'e'], 2, '4'), {'tools': ['e'], 'nextCursor': None}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert the cursor, loop to collect a page.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Convert the cursor, loop to collect a page.

```python
def page(tools, size, cursor):
    start = 0 if cursor is None else int(cursor)
    out = []
    i = start
    while i < len(tools) and len(out) < size:
        out.append(tools[i])
        i += 1
    return {"tools": out, "nextCursor": str(i) if i < len(tools) else None}
```

**Cost:** O(size) time, O(size) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing.

```python
def page(tools, size, cursor):
    start = int(cursor or 0)
    end = start + size
    return {"tools": tools[start:end], "nextCursor": str(end) if end < len(tools) else None}
```

**Cost:** O(size) time, O(size) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert the cursor, loop to collect a page. That costs O(size) time and O(size) space."

"I can do better: Slicing. That's O(size) time and O(size) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/paginate-tools-list)

<a id="q17"></a>
### Q17. Avoid Name Clashes Across Servers

🟡 Interview standard · *Namespacing*

A host connects to several MCP servers. Prefix every tool with its server (`server__tool`), and return `(sorted names, lookup)` where lookup maps each prefixed name back to `(server, tool)`.

**Examples**

- `servers = {'github': ['search', 'create_issue'], 'files': ['search', 'read']}` → `(['files__read', 'files__search', 'github__create_issue', 'github__search'], {'github__search': ('github', 'search'), 'github__create_issue': ('github', 'create_issue'), 'files__search': ('files', 'search'), 'files__read': ('files', 'read')})`

In [ ]:
def namespace(servers):
    """
    A host connects to several MCP servers. Prefix every tool with its server
    (server__tool), and return (sorted names, lookup) where lookup maps each
    prefixed name back to (server, tool).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(namespace, [
    (({'github': ['search', 'create_issue'], 'files': ['search', 'read']},), (['files__read', 'files__search', 'github__create_issue', 'github__search'], {'github__search': ('github', 'search'), 'github__create_issue': ('github', 'create_issue'), 'files__search': ('files', 'search'), 'files__read': ('files', 'read')})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops filling a list and a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One dict comprehension; the sorted keys are the names.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops filling a list and a dict.

```python
def namespace(servers):
    lookup = {}
    for server, tools in servers.items():
        for tool in tools:
            lookup[server + "__" + tool] = (server, tool)
    return (sorted(lookup), lookup)
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One dict comprehension; the sorted keys are the names.

```python
def namespace(servers):
    lookup = {f"{s}__{t}": (s, t) for s, tools in servers.items() for t in tools}
    return (sorted(lookup), lookup)
```

**Cost:** O(t log t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops filling a list and a dict. That costs O(t) time and O(t) space."

"I can do better: One dict comprehension; the sorted keys are the names. That's O(t log t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/avoid-name-clashes-across-servers)

<a id="q18"></a>
### Q18. Match Responses to Requests

🟡 Interview standard · *Out-of-order replies*

A client sent requests (id → method) and receives responses in any order, some of them errors. Return `{method: 'ok' or error message}` plus a list of ids still waiting, as `(results, pending)`.

**Examples**

- `sent = {1: 'tools/list', 2: 'tools/call', 3: 'resources/read'}, responses = [{'id': 2, 'result': {}}, {'id': 1, 'error': {'code': -32603, 'message': 'boom'}}]` → `({'tools/call': 'ok', 'tools/list': 'boom'}, [3])`

In [ ]:
def match(sent, responses):
    """
    A client sent requests (id → method) and receives responses in any order,
    some of them errors. Return {method: 'ok' or error message} plus a list of
    ids still waiting, as (results, pending).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(match, [
    (({1: 'tools/list', 2: 'tools/call', 3: 'resources/read'}, [{'id': 2, 'result': {}}, {'id': 1, 'error': {'code': -32603, 'message': 'boom'}}]), ({'tools/call': 'ok', 'tools/list': 'boom'}, [3])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over responses, look each id up, then find ids never answered.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension for results and set difference for pending.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over responses, look each id up, then find ids never answered.

```python
def match(sent, responses):
    results = {}
    answered = []
    for r in responses:
        method = sent[r["id"]]
        answered.append(r["id"])
        if "error" in r:
            results[method] = r["error"]["message"]
        else:
            results[method] = "ok"
    pending = []
    for i in sent:
        if i not in answered:
            pending.append(i)
    return (results, sorted(pending))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict comprehension for results and set difference for pending.

```python
def match(sent, responses):
    results = {sent[r["id"]]: r["error"]["message"] if "error" in r else "ok" for r in responses}
    return (results, sorted(set(sent) - {r["id"] for r in responses}))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over responses, look each id up, then find ids never answered. That costs O(n) time and O(n) space."

"I can do better: A dict comprehension for results and set difference for pending. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/match-responses-to-requests)

<a id="q19"></a>
### Q19. Decide Which Calls Need Approval

🟡 Interview standard · *Tool annotations · permissions*

Using MCP tool annotations, return `{tool: 'allow' | 'ask' | 'deny'}`: deny anything on the user's `blocked` list; allow read-only tools (`readOnlyHint` true); ask for everything else. Annotations are hints from the server, so a missing hint means 'ask'.

**Examples**

- `tools = [{'name': 'read_file', 'annotations': {'readOnlyHint': True}}, {'name': 'write_file', 'annotations': {'readOnlyHint': False, 'destructiveHint': False}}, {'name': 'delete_repo', 'annotations': {'destructiveHint': True}}, {'name': 'mystery', 'annotations': {}}], blocked = ['delete_repo']` → `{'read_file': 'allow', 'write_file': 'ask', 'delete_repo': 'deny', 'mystery': 'ask'}`
- `tools = [{'name': 'read_file', 'annotations': {'readOnlyHint': True}}, {'name': 'write_file', 'annotations': {'readOnlyHint': False, 'destructiveHint': False}}, {'name': 'delete_repo', 'annotations': {'destructiveHint': True}}, {'name': 'mystery', 'annotations': {}}], blocked = []` → `{'read_file': 'allow', 'write_file': 'ask', 'delete_repo': 'ask', 'mystery': 'ask'}`

In [ ]:
def policy(tools, blocked):
    """
    Using MCP tool annotations, return {tool: 'allow' | 'ask' | 'deny'}: deny
    anything on the user's blocked list; allow read-only tools (readOnlyHint
    true); ask for everything else. Annotations are hints from the server, so a
    missing hint means 'ask'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(policy, [
    (([{'name': 'read_file', 'annotations': {'readOnlyHint': True}}, {'name': 'write_file', 'annotations': {'readOnlyHint': False, 'destructiveHint': False}}, {'name': 'delete_repo', 'annotations': {'destructiveHint': True}}, {'name': 'mystery', 'annotations': {}}], ['delete_repo']), {'read_file': 'allow', 'write_file': 'ask', 'delete_repo': 'deny', 'mystery': 'ask'}),
    (([{'name': 'read_file', 'annotations': {'readOnlyHint': True}}, {'name': 'write_file', 'annotations': {'readOnlyHint': False, 'destructiveHint': False}}, {'name': 'delete_repo', 'annotations': {'destructiveHint': True}}, {'name': 'mystery', 'annotations': {}}], []), {'read_file': 'allow', 'write_file': 'ask', 'delete_repo': 'ask', 'mystery': 'ask'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain per tool.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension with a conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain per tool.

```python
def policy(tools, blocked):
    out = {}
    for t in tools:
        if t["name"] in blocked:
            out[t["name"]] = "deny"
        elif t["annotations"].get("readOnlyHint") is True:
            out[t["name"]] = "allow"
        else:
            out[t["name"]] = "ask"
    return out
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict comprehension with a conditional expression.

```python
def policy(tools, blocked):
    return {t["name"]: "deny" if t["name"] in blocked else "allow" if t["annotations"].get("readOnlyHint") is True else "ask"
            for t in tools}
```

**Cost:** O(t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain per tool. That costs O(t) time and O(t) space."

"I can do better: A dict comprehension with a conditional expression. That's O(t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/decide-which-calls-need-approval)

<a id="q20"></a>
### Q20. A Whole MCP Session

🔴 Stretch · *Putting it together*

Write a tiny server: `handle(msg)` answers `initialize` (with `protocolVersion` '2025-06-18' and `capabilities` {'tools': {}}), `tools/list` (names of REGISTRY tools, sorted) and `tools/call` (using the `tool_result` helper); notifications get no reply; anything else gets error -32601. Run the client `script` and return the list of replies' `result` or `error.code`.

**Examples**

- `script = [{'jsonrpc': '2.0', 'id': 1, 'method': 'initialize', 'params': {'protocolVersion': '2025-06-18'}}, {'jsonrpc': '2.0', 'method': 'notifications/initialized'}, {'jsonrpc': '2.0', 'id': 2, 'method': 'tools/list'}, {'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call', 'params': {'name': 'add', 'arguments': {'a': 1, 'b': 2}}}, {'jsonrpc': '2.0', 'id': 4, 'method': 'ping/pong'}]` → `[{'protocolVersion': '2025-06-18', 'capabilities': {'tools': {}}}, {'tools': ['add', 'get_weather']}, {'content': [{'type': 'text', 'text': '3'}], 'isError': False}, -32601]`

In [ ]:
def session(script):
    """
    Write a tiny server: handle(msg) answers initialize (with protocolVersion
    '2025-06-18' and capabilities {'tools': {}}), tools/list (names of REGISTRY
    tools, sorted) and tools/call (using the tool_result helper); notifications
    get no reply; anything else gets error -32601. Run the client script and
    return the list of replies' result or error.code.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(session, [
    (([{'jsonrpc': '2.0', 'id': 1, 'method': 'initialize', 'params': {'protocolVersion': '2025-06-18'}}, {'jsonrpc': '2.0', 'method': 'notifications/initialized'}, {'jsonrpc': '2.0', 'id': 2, 'method': 'tools/list'}, {'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call', 'params': {'name': 'add', 'arguments': {'a': 1, 'b': 2}}}, {'jsonrpc': '2.0', 'id': 4, 'method': 'ping/pong'}],), [{'protocolVersion': '2025-06-18', 'capabilities': {'tools': {}}}, {'tools': ['add', 'get_weather']}, {'content': [{'type': 'text', 'text': '3'}], 'isError': False}, -32601]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain inside a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dispatch table from method name to handler function.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain inside a loop.

```python
def tool_result(params):
    # the tools/call handler from earlier, as a plain helper the session can use
    try:
        value = REGISTRY[params["name"]](**params.get("arguments", {}))
        return {"content": [{"type": "text", "text": str(value)}], "isError": False}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"{type(e).__name__}: {e}"}], "isError": True}


def session(script):
    out = []
    for msg in script:
        if "id" not in msg:
            continue
        m = msg["method"]
        if m == "initialize":
            out.append({"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}})
        elif m == "tools/list":
            out.append({"tools": sorted(REGISTRY)})
        elif m == "tools/call":
            out.append(tool_result(msg["params"]))
        else:
            out.append(-32601)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dispatch table from method name to handler function.

```python
def tool_result(params):
    # the tools/call handler from earlier, as a plain helper the session can use
    try:
        value = REGISTRY[params["name"]](**params.get("arguments", {}))
        return {"content": [{"type": "text", "text": str(value)}], "isError": False}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"{type(e).__name__}: {e}"}], "isError": True}


def session(script):
    handlers = {"initialize": lambda p: {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}},
                "tools/list": lambda p: {"tools": sorted(REGISTRY)},
                "tools/call": tool_result}
    replies = []
    for msg in (m for m in script if "id" in m):
        h = handlers.get(msg["method"])
        replies.append(h(msg.get("params", {})) if h else -32601)
    return replies
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain inside a loop. That costs O(n) time and O(n) space."

"I can do better: A dispatch table from method name to handler function. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/mcp/a-whole-mcp-session)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What problem does MCP solve?</b></summary>

It standardises how AI applications connect to tools and data, so a connector written once as an MCP server works with every MCP host — M + N integrations instead of M × N.

</details>

<details>
<summary><b>2. Tools vs resources vs prompts?</b></summary>

Tools are actions the model chooses to call; resources are data the application reads into context by URI; prompts are templates the user selects. They differ in who is in control.

</details>

<details>
<summary><b>3. When would you use stdio vs HTTP transport?</b></summary>

stdio for local servers the host launches itself (files, local databases, developer tools); Streamable HTTP for remote servers shared by many users, with OAuth authorisation.

</details>

<details>
<summary><b>4. How should a tool report a failure?</b></summary>

Return a normal result with isError true and a helpful message, so the model can read it and recover. Use JSON-RPC errors only for malformed requests or unknown methods.

</details>

<details>
<summary><b>5. What are the main security risks of MCP?</b></summary>

Tools act with the user's permissions, tool output can carry prompt injections, and local servers run code on the machine. Mitigate with approval for side effects, least-privilege credentials, trusted servers only and treating tool output as untrusted data.

</details>

<details>
<summary><b>6. How do you design a good tool?</b></summary>

A clear name and description that says when to use it, a small typed input schema with enums, idempotent behaviour where possible, and short, structured output that fits the model's context.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why MCP exists
- [ ] I can explain: The wire format: JSON-RPC 2.0
- [ ] I can explain: The lifecycle: initialize, then work
- [ ] I can explain: Tools: schemas, calls and results
- [ ] I can explain: Build a real server in a few lines
- [ ] I can explain: Hosts, many servers, and security

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Paginate tools/list
- [ ] [Q17](#q17) Avoid Name Clashes Across Servers
- [ ] [Q18](#q18) Match Responses to Requests
- [ ] [Q19](#q19) Decide Which Calls Need Approval
- [ ] [Q20](#q20) A Whole MCP Session

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.